# PPO RL: pricing model train

Simulator set-up as in the newsvendor notebooks (demand, inventory, suppliers, forecast,
newsvendor replenishment). Prices are set every day by one PPO agent per department,
starting from the pretrained checkpoints in `RL/models/`.

In [12]:
import sys
import importlib
from pathlib import Path
from collections import defaultdict, deque

import numpy as np
import polars as pl
import scipy
import cvxpy as cp
import torch as T
from tqdm.auto import tqdm

sys.path.insert(0, str(Path.cwd().parent / 'sim'))     # sim.py
sys.path.insert(0, str(Path.cwd() / 'RL'))             # PPO.py, scaler/

from sim import Demand, Env, Inventory, SupplierInfo, Supplier
import PPO; importlib.reload(PPO)
from scaler.custom_scaler import CustomScaler

## Data

M5 sales and prices for stores CA_1 + CA_2, rolled up to department x day.

In [13]:
data_folder = Path.cwd().parents[1] / 'data'

# --- ingestion: no data is read here; scan_csv only inspects the header ---
calendar_lf = pl.scan_csv(data_folder / 'calendar.csv')
prices_lf   = pl.scan_csv(data_folder / 'sell_prices.csv')
sales_lf    = pl.scan_csv(data_folder / 'sales_train_evaluation.csv')

In [14]:
STORE = ['CA_1', 'CA_2']

# collect_schema() reads the header only, it does not execute the plan.
day_cols  = [c for c in sales_lf.collect_schema().names() if c.startswith('d_')]
day_order = sorted(day_cols, key=lambda c: int(c.removeprefix('d_')))   # chronological, not lexical

# --- demand: wide daily matrix -> long (item_id, d, demand) + financial week ---
demand_lf = (
    sales_lf
    .filter(pl.col('store_id').is_in(STORE))          # predicate pushed into the scan
    .select(['item_id', *day_cols])
    .unpivot(index='item_id', on=day_cols, variable_name='d', value_name='demand')
    .join(calendar_lf.select('d', 'wm_yr_wk'), on='d', how='inner')
)

# --- prices: same store, join keys + price only (projection pushdown drops the rest) ---
prices_store_lf = (
    prices_lf
    .filter(pl.col('store_id').is_in(STORE))
    .select('item_id', 'wm_yr_wk', 'sell_price')
) 

# --- item-day fact table ---
merged_lf = (
    demand_lf
    .join(prices_store_lf, on=['item_id', 'wm_yr_wk'], how='inner')   # inner: drops unpriced item-weeks
    .with_columns(
        pl.col('item_id').str.split('_').list.slice(0, 2).list.join('_').alias('dept_id'),
        pl.col('d').str.split('_').list.get(-1).cast(pl.Int32).alias('day_n'),
    )
)

# --- department x day aggregate ---
dept_day_lf = (
    merged_lf
    .group_by('dept_id', 'day_n')
    .agg(
        pl.col('demand').sum().alias('total_demand'),
        (pl.col('demand') * pl.col('sell_price')).sum().alias('total_weighted_price'),
    )
    .with_columns(
        (pl.col('total_weighted_price') / pl.col('total_demand')).round(3).alias('price_avg')
    )
    .filter(pl.col('price_avg').is_not_nan())     # zero-demand department-days give 0/0
)

day_spine = pl.LazyFrame(
    {'day_n': [int(c.removeprefix('d_')) for c in day_order]},
    schema={'day_n': pl.Int32},
)

wide_lf = (
    dept_day_lf.select('dept_id').unique()
    .join(day_spine, how='cross')                                     # every dept x every day
    .join(dept_day_lf.select('dept_id', 'day_n', 'total_demand'),
          on=['dept_id', 'day_n'], how='left')
    .with_columns(pl.col('total_demand').fill_null(0))
    .group_by('dept_id')
    .agg(pl.col('total_demand').sort_by('day_n'))                     # explicit ordering
    .with_columns(pl.col('total_demand').list.to_struct(fields=day_order))
    .unnest('total_demand')
    .sort(by='dept_id')
)

wide_df = wide_lf.collect()          # .collect(engine="streaming") if memory is tight

# department x day table with the average price, for the WTP curves
ca_1 = dept_day_lf.collect()

## Demand

In [15]:
wtp_kde = {
    df['dept_id'][0]: scipy.stats.gaussian_kde(df['price_avg'], weights=df['total_demand'])
    for df in ca_1.partition_by('dept_id')
}

In [16]:
demand = Demand(wide_df, wtp_kde)

## Environment, inventory and suppliers

In [17]:
SEED = 42
sim = Env(seed=SEED)

In [18]:
initial_order = {
    'FOODS_1': {
        'quantity': 270,
        'price': 2.5,   
        'shelf_life': 60,
    },
    'FOODS_2': {
        'quantity': 5000,
        'price': 3.3,   
        'shelf_life': 120,
    },
    'FOODS_3': {
        'quantity': 5000,
        'price': 2.3,   
        'shelf_life': 10,
    },
    'HOBBIES_1': {
        'quantity': 5000,
        'price': 3.7,   
        'shelf_life': 1000,
    },
    'HOBBIES_2': {
        'quantity': 5000,
        'price': 2.3,   
        'shelf_life': 1000,
    },
    'HOUSEHOLD_1': {
        'quantity': 5000,
        'price': 3.9,   
        'shelf_life': 1000,
    },
    'HOUSEHOLD_2': {
        'quantity': 5000,
        'price': 5.2,   
        'shelf_life': 1000,
    },
}

In [19]:
inventory = Inventory(sim, demand.wtp)

In [20]:
supplier_book = Supplier()

DEPT_DATA = {
    'FOODS_3':     dict(price_mean=2.33, price_std=0.16, lead_min=1, lead_max=4,
                         low =dict(q=[65, 215, 640], margin=0.35),
                         prem=dict(q=[60, 200, 610], margin=0.20)),
    'FOODS_1':     dict(price_mean=2.30, price_std=0.20, lead_min=5, lead_max=10,
                         low =dict(q=[25, 85, 250], margin=0.45),
                         prem=dict(q=[25, 80, 230], margin=0.30)),
    'FOODS_2':     dict(price_mean=3.31, price_std=0.27, lead_min=3, lead_max=7,
                         low =dict(q=[17, 60, 170], margin=0.30),
                         prem=dict(q=[17, 55, 170], margin=0.20)),
    'HOBBIES_1':   dict(price_mean=3.63, price_std=0.76, lead_min=5, lead_max=10,
                         low =dict(q=[30, 100, 300], margin=0.40),
                         prem=dict(q=[20, 75, 250], margin=0.30)),
    'HOBBIES_2':   dict(price_mean=2.19, price_std=0.41, lead_min=7, lead_max=14,
                         low =dict(q=[20, 100, 200], margin=0.50),
                         prem=dict(q=[20, 100, 200], margin=0.35)),
    'HOUSEHOLD_1': dict(price_mean=3.87, price_std=0.27, lead_min=7, lead_max=14,
                         low =dict(q=[25, 85, 250], margin=0.38),
                         prem=dict(q=[25, 75, 230], margin=0.28)),
    'HOUSEHOLD_2': dict(price_mean=5.18, price_std=0.23, lead_min=5, lead_max=10,
                         low =dict(q=[50, 170, 500], margin=0.30),
                         prem=dict(q=[50, 160, 480], margin=0.20)),
}

TIER_MULTIPLIERS = [1.0, 0.5, 0.2]  # p1, p2, p3 relative to lot price at min order qty


def build_price_dists(price_mean, price_std, margin):
    cv = price_std / price_mean  # constant coefficient of variation across tiers
    lot_price = price_mean * (1 - margin)
    return [
        scipy.stats.norm(loc=lot_price * m, scale=lot_price * m * cv)
        for m in TIER_MULTIPLIERS
    ]


for dept, cfg in DEPT_DATA.items():
    lead_time_low = scipy.stats.expon(loc=cfg['lead_max'], scale=cfg['lead_max'])
    lead_time_prem = scipy.stats.expon(loc=cfg['lead_min'], scale=cfg['lead_min'])

    supplier_book.add_supplier(
        SupplierInfo(
            name=f'{dept}_low',
            product=dept,
            price=build_price_dists(cfg['price_mean'], cfg['price_std'], cfg['low']['margin']),
            lead_time=lead_time_low,
            order_quantity=cfg['low']['q'],
        ),
        SupplierInfo(
            name=f'{dept}_premium',
            product=dept,
            price=build_price_dists(cfg['price_mean'], cfg['price_std'], cfg['prem']['margin']),
            lead_time=lead_time_prem,
            order_quantity=cfg['prem']['q'],
        ),
    )

## Simulation settings and forecast

In [21]:
EPOCH = 100
RUN_DAYS = 1900

shelf_life_dict = {
    'FOODS_1'       : 270,
    'FOODS_2'       : 120,
    'FOODS_3'       : 12,
    'HOBBIES_1'     : 500,
    'HOBBIES_2'     : 500,
    'HOUSEHOLD_1'   : 400,
    'HOUSEHOLD_2'   : 400,
}
wastage = []

In [22]:
FORECAST_WINDOW = 30

# Per-product forecast accuracy, in demand.product order. Frozen normals rather than fixed
# numbers: forecast quality is itself uncertain, so every call draws a fresh MAPE and the
# long-run average per department is its loc. Keep scale well under loc — draws are floored
# at 0, so a wide scale piles probability mass on that floor (i.e. perfect forecasts).
FORECAST_MAPE_BY_DEPT = {
    'FOODS_1':     scipy.stats.norm(loc=0.20, scale=0.03),
    'FOODS_2':     scipy.stats.norm(loc=0.15, scale=0.03),
    'FOODS_3':     scipy.stats.norm(loc=0.10, scale=0.03),
    'HOBBIES_1':   scipy.stats.norm(loc=0.15, scale=0.03),
    'HOBBIES_2':   scipy.stats.norm(loc=0.25, scale=0.03),
    'HOUSEHOLD_1': scipy.stats.norm(loc=0.10, scale=0.03),
    'HOUSEHOLD_2': scipy.stats.norm(loc=0.15, scale=0.03),
}


def _draw_mape(value, rng):
    """A frozen scipy distribution becomes a draw; a plain number passes through.

    Floored at 0 because a MAPE cannot be negative, and a normal always can dip below it.
    """
    if hasattr(value, 'rvs'):
        return max(0.0, float(value.rvs(random_state=rng)))

    return float(value)


def forecast(env, data, product, mape=FORECAST_MAPE_BY_DEPT, window=FORECAST_WINDOW,
             seed=None, products=None, history=False):
    """Forecast `product`'s daily demand for each of the next `window` days.

    `data` : (days, products) actual demand, i.e. generate_demand().T
    `product` : which column of `data` to forecast.
    `mape` : a number, a frozen scipy distribution, or a dict of either keyed by product
             (the default). Distributions are drawn fresh on every call, so accuracy varies
             day to day rather than being pinned.
    `products` : column order of `data`; defaults to demand.product.
    `seed` : int or np.random.Generator for the forecast errors; defaults to env.forecast_rng,
             so forecasts follow the simulation's seed.
    `history` : if truthy, also return all actual demand strictly before env.clock,
                oldest first (today is not included); if falsy, that part is empty.

    Returns (past, forecast): the forecast has one value per forecast day (fewer near
    the end of the demand path). The
    per-day relative errors are rescaled so their mean over the window — the MAPE — lands
    exactly on the drawn target. Days with zero actual demand are left out of that mean,
    since a percentage error is undefined there, and are forecast as zero.
    """
    rng = np.random.default_rng(env.forecast_rng if seed is None else seed)
    data = np.asarray(data)
    n_days = data.shape[0]

    products = list(products if products is not None else demand.product)
    col = products.index(product)

    # Forecast starts tomorrow, truncated by the end of the demand path. min() rather than
    # clamping both ends to n_days-1, which collapsed to an empty slice on the last days
    # and silently returned a zero forecast.
    start = min(env.clock + 1, n_days)
    end = min(start + window, n_days)

    y_true = data[start:end, col].astype(float)

    if isinstance(mape, dict):
        mape = mape[product]
    target = _draw_mape(mape, rng)

    # Shape of the magnitudes is arbitrary — the rescale below is what fixes the MAPE.
    # Only days with nonzero actuals count toward MAPE, so rescale over those alone.
    scored = y_true > 0
    magnitudes = rng.uniform(0.1, 1.0, size=y_true.size)
    if scored.any():
        magnitudes = magnitudes * (target / magnitudes[scored].mean())

    signs = rng.choice([-1.0, 1.0], size=y_true.size)
    y_pred = y_true * (1 + signs * magnitudes)

    # Demand cannot go negative; only reachable if a day's magnitude exceeds 1.0, i.e. a
    # target MAPE above roughly 0.55. Clipping there would pull the MAPE off target.
    past = data[:env.clock, col] if history else data[:0, col]

    return past, np.round(y_pred)

## Policy: newsvendor replenishment (prices are set by the PPO agents)

In [ ]:
LEAD_WINDOW = 5   # realised lead times kept per product, pooled over its suppliers


class Policy:

    def __init__(self, inventory, z=1.65):
        self.z = z   # safety factor for the reorder point
        self.reorder_point = {}
        for product in demand.product:
            self.reorder_point[product] = 0

        self.on_order = defaultdict(int)    # product units in transit
        self.lead_obs = defaultdict(lambda: deque(maxlen=LEAD_WINDOW))  # product last lead times

        self.markup = {
                        'FOODS_1': 0.4,
                        'FOODS_2': 0.25,
                        'FOODS_3': 0.3,
                        'HOBBIES_1': 0.35,
                        'HOBBIES_2': 0.4,
                        'HOUSEHOLD_1': 0.32,
                        'HOUSEHOLD_2': 0.25}

        # Decision horizon (days an order should cover) per product. Non-perishables are
        # capped at 100 days rather than their 1000-day shelf life, so a newsvendor order
        # covers a sensible cycle instead of years of demand. FOODS_1 is 60 to match its
        # opening lot's shelf life in initial_order, so the opening stock can sell before
        # it expires.
        self.shelf_life = {
                        'FOODS_1': 60,
                        'FOODS_2': 120,
                        'FOODS_3': 10,
                        'HOBBIES_1': 500,
                        'HOBBIES_2': 500,
                        'HOUSEHOLD_1': 400,
                        'HOUSEHOLD_2': 400}

        # Lead-time prior per product (used until LEAD_WINDOW deliveries are observed): an
        # equal mix of its suppliers' lead-time distributions. The distributions are fixed, so
        # this is computed once rather than on every daily reorder-point update.
        self.lead_prior = {}
        for product in demand.product:
            dists = [s.lead_time for s in supplier_book.by_product[product]]
            means = np.array([dist.mean() for dist in dists])
            variances = np.array([dist.var() for dist in dists])
            self.lead_prior[product] = (means.mean(), np.sqrt(variances.mean() + means.var()))

        self.unit_cost = {}   # product -> unit cost of the latest order

    def reset_epoch(self):
        """Clear stock in transit; the calendar is wiped on sim.reset(). Lead-time history is kept."""
        self.on_order.clear()
        self.unit_cost.clear()

    def record_order(self, placed):
        for o in placed:
            self.on_order[o['product']] += o['quantity']

    def record_arrival(self, product, quantity, lead_time):
        self.on_order[product] -= quantity
        self.lead_obs[product].append(lead_time)

    def lead_stats(self, product):
        """Mean and std of the product's lead time over its last LEAD_WINDOW deliveries,
        whichever supplier shipped them.

        Until the window is full, falls back to an equal mix of the product's suppliers'
        lead-time distributions: mean of the means, and the mixture std, which adds the
        spread between suppliers to each one's own variance.
        """
        obs = self.lead_obs[product]
        if len(obs) == LEAD_WINDOW:
            return np.mean(obs), np.std(obs, ddof=1)
        return self.lead_prior[product]

    def update_reorder_point(self, product):
        """d*E[L] + z*sigma_LT, with demand and the product's lead-time uncertainty."""
        _, fc = forecast(sim, demand_data, product, window=self.shelf_life[product])
        d, sigma = np.mean(fc), np.std(fc)

        L, sL = self.lead_stats(product)
        sigma_lt = np.sqrt(L * sigma**2 + d**2 * sL**2)   # demand noise + lead-time noise
        self.reorder_point[product] = d * L + self.z * sigma_lt

    def newsvendor(self, product, quotes):
        decision_period = self.shelf_life[product]

        quote_price = []
        for quote in quotes:
            quote_price += list(quote.price)

        max_cost = np.max(quote_price)
        max_price = max_cost*(1+self.markup[product]) 
        historical_data, forecast_data = forecast(sim, demand_data, product, window=decision_period, history=True)


        cost_u, cost_o = (max_price-max_cost), max_cost
        critical_ratio = (cost_u)/(cost_u+cost_o)

        # Non-overlapping shelf-life windows; drop the partial window at the end so reshape works.
        n_windows = len(historical_data) // decision_period
        if n_windows == 0:
            # Not even one full window of history yet: fall back to the forecast total.
            return np.round(np.sum(forecast_data))

        historical_data = np.asarray(historical_data[:n_windows * decision_period])
        window_demand = historical_data.reshape(-1, decision_period).sum(axis=1)
        D = np.round(np.percentile(window_demand, critical_ratio*100))

        return D

    def order_lp(self, product, quotes, order_qty):
        """Pick one supplier and how many lots of each size to buy from it.

        Returns (quote, lots) where lots[t] is the number of lots of quote.order_quantity[t],
        or None if the solve fails. order_purchase takes one lot per call, so place each lot
        separately.
        """
        decision_period = self.shelf_life[product]
        _, forecast_data = forecast(sim, demand_data, product, window=decision_period)

        d, sigma = np.mean(forecast_data), max(np.std(forecast_data), 1e-6) # demand rate, variability

        cost = np.array([q.price for q in quotes], dtype=float)          # (suppliers, tiers) unit price
        lot  = np.array([q.order_quantity for q in quotes], dtype=float)  # (suppliers, tiers) lot size
        # Each supplier's own lead-time distribution, so the choice still reflects which one
        # is faster (observed lead times are pooled by product and can't tell them apart).
        # Not the quote's lead_time: that is the exponential's loc, i.e. its minimum.
        lead_mean = np.array([q.supplier.lead_time.mean() for q in quotes], dtype=float)
        lead_std  = np.array([q.supplier.lead_time.std() for q in quotes], dtype=float)

        margin = np.max(cost)*(self.markup[product])  # profit lost per unit short
        cost_o = np.max(cost)                         # cost of a unit left over (as in newsvendor)

        # Expected units lost before each supplier's delivery lands (normal loss function).
        # Lead-time demand variance = demand noise over L days + lead-time noise at rate d.
        d_l = d*lead_mean
        sigma_l = np.sqrt(lead_mean*sigma**2 + d**2*lead_std**2)
        position = inventory.inventory_status().get(product, {'quantity': 0})['quantity'] # quantity in stock

        k = (position-d_l)/sigma_l
        shortfall = sigma_l * (scipy.stats.norm.pdf(k) - k*scipy.stats.norm.sf(k))   # (suppliers,)

        x = cp.Variable(lot.shape, integer=True, nonneg=True)   # lots per (supplier, tier)
        y = cp.Variable(len(quotes), boolean=True)              # 1 for the chosen supplier
        under = cp.Variable(integer=True, nonneg=True)              # units short of order_qty
        over  = cp.Variable(integer=True, nonneg=True)              # units above order_qty

        qty = cp.sum(cp.multiply(lot, x))             # total units ordered

        constraints = [
            cp.sum(y) == 1,                                          # one supplier
            x <= 100 * cp.reshape(y, (len(quotes), 1), order='C'),   # lots only from the chosen supplier
            qty + under - over == order_qty,          # gap to the newsvendor quantity
        ]
        obj = (cp.sum(cp.multiply(cost * lot, x))     # purchase cost
               + margin * (shortfall @ y)             # lost sales while waiting for delivery
               + margin * under                       # lot too small for the cycle
               + cost_o * over)                       # lot too big, risk of waste
        prob = cp.Problem(cp.Minimize(obj), constraints)
        prob.solve(solver=cp.HIGHS)

        if prob.status != cp.OPTIMAL:
            return None
        s = int(np.argmax(y.value))
        return quotes[s], np.round(x.value[s]).astype(int)

    def decision(self, inventory, **karg):
        """Reorder any product at or below its reorder point; returns the orders placed today.

        Prices are not set here: the PPO agents set them every day."""
        placed_today = []
        status = inventory.inventory_status()
        for product in self.reorder_point:
            # inventory_status only lists products currently in stock
            on_hand = status.get(product, {'quantity': 0})['quantity']
            position = on_hand + self.on_order[product]     # count stock in transit

            self.update_reorder_point(product)              # demand and lead times move daily
            if position <= self.reorder_point[product]:
                quotes = supplier_book.order_proposal(product, rng=sim.supplier_rng)
                order_qty = self.newsvendor(product, quotes)

                result = self.order_lp(product, quotes, order_qty)
                if result is None:
                    continue
                quote, lots = result    # lots[t] = number of lots of quote.order_quantity[t]

                # Every lot size in one order: a single delivery, one lead-time draw.
                placed = supplier_book.order_purchase(
                    sim,
                    {quote.supplier: {'lots': [int(n) for n in lots], 'price': quote.price}},
                    demand_index=0,
                )
                self.record_order(placed)
                placed_today.extend(placed)

                # Unit cost of this order (weighted by units) becomes the price floor.
                if placed:
                    self.unit_cost[product] = placed[0]['unit_price']

        return placed_today


policy = Policy(inventory)

## Simulator steps

`reset_world` starts a run, `begin_day` handles deliveries and expiries, `finish_day` sets the
agents' prices, sells the day's demand, reorders and advances the clock.

In [24]:
def process_events(time, event):
    if event[0] == 'stockup':
        _, product, quantity, price, supplier_name, ordered_on = event
        inventory.stockup({product: {'quantity': quantity, 'shelf_life': shelf_life_dict[product]}})
        policy.record_arrival(product, quantity, lead_time=time - ordered_on)
        return None
    if event[0] == 'expire':
        flushout = inventory.flushout_expired(event[1])
        wastage.append((time, flushout))
        return flushout     # [products, units written off]


def reset_world(seed):
    """New run: demand path and opening stock (newsvendor quantity, initial_order prices)."""
    global demand_data                      # forecast() and Policy read this global
    sim.reset(seed=seed)
    inventory.reset(keep_price=False)
    policy.reset_epoch()
    demand_data = demand.generate_demand(seed=sim.demand_rng).T.astype(int)
    opening_order = {
        p: {'quantity': int(policy.newsvendor(p, supplier_book.order_proposal(p, rng=sim.supplier_rng))),
            'price': initial_order[p]['price'],
            'shelf_life': initial_order[p]['shelf_life']}
        for p in demand.product
    }
    inventory.stockup(opening_order)
    inventory.set_price(opening_order)


def begin_day():
    """Today's deliveries and expiries; returns the units written off per product."""
    day_wastage = defaultdict(int)
    if sim.clock in sim.event_calendar:
        for event in sim.event_calendar[sim.clock]:
            flushout = process_events(sim.clock, event)
            if event[0] == 'expire':
                for product, units in zip(*flushout):
                    day_wastage[product] += units
        sim.remove_event(sim.clock)
    return day_wastage


def finish_day(prices, day_wastage):
    """Set prices, sell today's demand, reorder, advance the clock. Returns today's record per product."""
    if prices:
        inventory.set_price({p: {'price': float(v)} for p, v in prices.items()})
    day = sim.clock
    todays_demand = demand_data[day]
    todays_price = {p: info['price'] for p, info in inventory.inventory_status().items()}
    sold = inventory.sell_stock({p: int(todays_demand[i]) for i, p in enumerate(demand.product)})

    order_qty, order_cost = defaultdict(int), defaultdict(float)
    for order in policy.decision(inventory):
        order_qty[order['product']] += order['quantity']
        order_cost[order['product']] += order['cost']

    status = inventory.inventory_status()
    rec = {}
    for i, p in enumerate(demand.product):
        ordered = p in order_qty
        rec[p] = {'day': day, 'demand': int(todays_demand[i]), 'sold': sold.get(p, 0),
                  'price': float(todays_price.get(p, np.nan)), 'wastage': day_wastage[p],
                  'order_qty': order_qty[p] if ordered else None,
                  'order_cost': float(order_cost[p]) if ordered else None,
                  'stock': status.get(p, {}).get('quantity', 0)}
    sim.clock_step()
    return rec


def unit_cost(p):
    """Unit cost of the latest order; before the first order, the opening price less the markup."""
    c = policy.unit_cost.get(p)
    return c if c is not None else initial_order[p]['price'] / (1 + policy.markup[p])

## PPO agents

One agent per department, loaded from the pretrained checkpoints. Fine-tuned weights are
saved to `RL/models_ppo/` so the pretrained ones are kept.

In [25]:
scaler   = CustomScaler()                                   # same settings as in pretraining
products = list(demand.product)

PRETRAINED_DIR = Path.cwd() / 'RL' / 'models'
FINETUNED_DIR  = Path.cwd() / 'RL' / 'models_ppo'
ACTOR_DIMS, CRITIC_DIMS = (64, 64), (128, 128)              # must match the pretrained checkpoints
GAMMA        = 0.99
EPISODE_DAYS = 365                                          # days per training episode
N_STEPS      = 730                                          # days collected per PPO update
N_EPISODES   = 200

agents = {}
for p in products:
    agent = PPO.Agent(n_actions=1, input_dims=3, action_low=[-1.0], action_high=[1.0],
                      gamma=GAMMA, alpha=1e-4, gae_lambda=0.95, batch_size=256, n_epochs=10,
                      entropy_coef=0.001, actor_dims=ACTOR_DIMS, critic_dims=CRITIC_DIMS,
                      save_dir=PRETRAINED_DIR / p)
    agent.load_models()
    agent.actor.checkpoint_file  = FINETUNED_DIR / p / 'actor_torch_ppo'
    agent.critic.checkpoint_file = FINETUNED_DIR / p / 'critic_torch_ppo'
    agents[p] = agent

loading models.....
loading models.....
loading models.....
loading models.....
loading models.....
loading models.....
loading models.....


In [26]:
def to_prev(rec, prev=None, prices=None):
    """Yesterday's price, demand and closing stock; keeps the set price if the product sold out."""
    out = {}
    for p in products:
        price = rec[p]['price']
        if not np.isfinite(price):
            price = prices[p] if prices else prev[p]['price']
        out[p] = {'price': price, 'demand': rec[p]['demand'], 'stock': rec[p]['stock']}
    return out


def observe(p, prev):
    """Scaled [yesterday's price, demand, closing stock], as in pretraining."""
    return scaler.input_scaler(p, [prev[p]['price'], prev[p]['demand'], prev[p]['stock']])


def reward_of(p, r):
    """Scaled revenue minus scaled waste cost, as in pretraining."""
    price = r['price'] if np.isfinite(r['price']) else 0.0
    return float(scaler.output_scaler(p, [price * r['sold'], unit_cost(p) * r['wastage']]).sum())


def value_of(agent, obs):
    with T.no_grad():
        return agent.critic(T.tensor(obs, dtype=T.float32, device=agent.critic.device)).item()


def start_episode(seed):
    """Reset and play day 0 at the opening prices (it provides "yesterday")."""
    reset_world(seed)
    return to_prev(finish_day(None, begin_day()))

## Train

In [27]:
rng, step, history = np.random.default_rng(0), 0, []

for ep in tqdm(range(N_EPISODES), desc='episodes'):
    prev = start_episode(int(rng.integers(1_000, 1_000_000)))   # not the evaluation seeds
    ep_reward = dict.fromkeys(products, 0.0)

    while True:
        day_wastage = begin_day()
        obs = {p: observe(p, prev) for p in products}
        out = {p: agents[p].choose_actions(obs[p]) for p in products}      # (clipped, raw, logp, value)
        prices = {p: float(scaler.price_inverse(p, out[p][0][0])) for p in products}

        rec = finish_day(prices, day_wastage)
        truncated = sim.clock > EPISODE_DAYS
        prev = to_prev(rec, prev, prices)

        for p in products:
            _, raw, logp, v = out[p]
            r = reward_of(p, rec[p])
            ep_reward[p] += r
            if truncated:                                   # time limit, not a real end: bootstrap
                r += GAMMA * value_of(agents[p], observe(p, prev))
            agents[p].remember(obs[p], raw, logp, v, r, truncated)
        step += 1

        if step % N_STEPS == 0:
            for p in products:
                agents[p].learn(last_value=value_of(agents[p], observe(p, prev)))
        if truncated:
            break

    history.append({'episode': ep, **ep_reward})
    if (ep + 1) % 10 == 0:
        recent = pl.DataFrame(history[-10:]).drop('episode').mean()
        print(f'episode {ep+1} | mean reward (last 10):', {p: round(recent[p][0], 1) for p in products})
        for p in products:
            agents[p].save_models()

episodes:   0%|          | 0/200 [00:00<?, ?it/s]

episode 10 | mean reward (last 10): {'FOODS_1': 34.8, 'FOODS_2': 15.5, 'FOODS_3': 67.3, 'HOBBIES_1': 16.6, 'HOBBIES_2': 4.9, 'HOUSEHOLD_1': 87.1, 'HOUSEHOLD_2': 145.8}
saving models.....
saving models.....
saving models.....
saving models.....
saving models.....
saving models.....
saving models.....
episode 20 | mean reward (last 10): {'FOODS_1': 44.0, 'FOODS_2': 17.8, 'FOODS_3': 83.3, 'HOBBIES_1': 17.1, 'HOBBIES_2': 5.0, 'HOUSEHOLD_1': 93.5, 'HOUSEHOLD_2': 154.8}
saving models.....
saving models.....
saving models.....
saving models.....
saving models.....
saving models.....
saving models.....
episode 30 | mean reward (last 10): {'FOODS_1': 47.5, 'FOODS_2': 19.1, 'FOODS_3': 91.7, 'HOBBIES_1': 17.3, 'HOBBIES_2': 5.0, 'HOUSEHOLD_1': 90.5, 'HOUSEHOLD_2': 153.9}
saving models.....
saving models.....
saving models.....
saving models.....
saving models.....
saving models.....
saving models.....
episode 40 | mean reward (last 10): {'FOODS_1': 46.8, 'FOODS_2': 20.9, 'FOODS_3': 86.4, 'HOBBIES_

## Evaluate

Same seeds (0..EPOCH-1) and days as the newsvendor notebooks, mean action (no exploration).
Writes `results/Newsvendor_SingleSupplier_PPO.parquet` for `compare_performance.ipynb`.

In [28]:
def evaluate(seeds=np.arange(EPOCH), days=RUN_DAYS):
    log = defaultdict(list)
    for epoch, seed in enumerate(tqdm(seeds, desc='evaluation')):
        prev = None
        reset_world(int(seed))
        while sim.clock <= days:
            day_wastage = begin_day()
            prices = None                                   # day 0 at the opening prices
            if prev is not None:
                prices = {}
                for p in products:
                    s = T.tensor(observe(p, prev), dtype=T.float32, device=agents[p].actor.device).unsqueeze(0)
                    with T.no_grad():
                        a = agents[p].actor(s).mean.item()
                    prices[p] = float(scaler.price_inverse(p, np.clip(a, -1, 1)))
            rec = finish_day(prices, day_wastage)
            prev = to_prev(rec, prev, prices)
            for p in products:
                log['epoch'].append(epoch); log['seed'].append(int(seed)); log['product'].append(p)
                for k in ('day', 'demand', 'sold', 'price', 'wastage', 'order_qty', 'order_cost', 'stock'):
                    log[k].append(rec[p][k])

    return (pl.DataFrame(log, schema={
                'epoch': pl.Int32, 'seed': pl.Int64, 'day': pl.Int32, 'product': pl.Utf8,
                'demand': pl.Int64, 'sold': pl.Int64, 'price': pl.Float64, 'wastage': pl.Int64,
                'order_qty': pl.Int64, 'order_cost': pl.Float64, 'stock': pl.Int64})
            .select('epoch', 'seed', 'day', 'product', 'demand', 'sold', 'price',
                    'wastage', 'order_qty', 'order_cost', 'stock')
            .with_columns(pl.col('price').fill_nan(None)))


ppo_log = evaluate()
results_folder = Path.cwd().parents[1] / 'results'
results_folder.mkdir(exist_ok=True)
ppo_log.write_parquet(results_folder / 'Newsvendor_SingleSupplier_PPO.parquet')
ppo_log

evaluation:   0%|          | 0/100 [00:00<?, ?it/s]

epoch,seed,day,product,demand,sold,price,wastage,order_qty,order_cost,stock
i32,i64,i32,str,i64,i64,f64,i64,i64,f64,i64
0,0,0,"""FOODS_1""",1305,703,2.5,0,null,null,69918
0,0,0,"""FOODS_2""",1073,602,3.3,0,null,null,95974
0,0,0,"""FOODS_3""",7602,3590,2.3,0,null,null,55779
0,0,0,"""HOBBIES_1""",1798,894,3.7,0,null,null,668726
0,0,0,"""HOBBIES_2""",36,14,2.3,0,null,null,34535
…,…,…,…,…,…,…,…,…,…,…
99,99,1900,"""FOODS_3""",5779,5777,1.76,0,null,null,54634
99,99,1900,"""HOBBIES_1""",1437,856,3.48,0,null,null,45199
99,99,1900,"""HOBBIES_2""",142,141,1.15,0,null,null,20377
